# Dataset Analysis
This notebook contains the analysis of the dataset, including data cleaning, exploration, and visualization. The goal is to understand the structure of the data and identify any patterns or insights that can be derived from it.

### Library Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import folium
from folium.plugins import HeatMap

### Reading the Dataset from the CSV file

In [ ]:
data_dir = "../data/"
csv_files = list(Path(data_dir).glob("*.csv"))

# Read all CSV files into a list of DataFrames
dfs = [pd.read_csv(file) for file in csv_files]
# Concatenate all DataFrames into a single DataFrame
df = pd.concat(dfs, ignore_index=True)
df.head()

## Dataset Overview
First, we are gonna analyze the dataset to understand its structure, identify any missing values, and get a general sense of the data.

In [ ]:
df.info()

# Check all the possible values for rideable_type
print("\nUnique values for rideable_type: ", df["rideable_type"].unique().tolist())

# Check all the possible values for member_casual
print("\nUnique values for member_casual: ", df["member_casual"].unique().tolist())

# Check how many unique values for stations
unique_stations = pd.concat([df["start_station_name"], df["end_station_name"]]).nunique()
print(f"\nNumber of unique stations: {unique_stations}")

### Check missing values

In [ ]:
# Check the entries where end station name is missing
missing_end_station = df[df["end_station_name"].isna()]
#print(f"\nEntries with missing end station names: {len(missing_end_station)}")
#print(missing_end_station.head())

# We can safely drop the rows with missing end station names, as they won't contribute to our analysis of station flows
df = df.dropna(subset=["end_station_name"])

missing_end_station_id = df[df["end_station_id"].isna()]
#print(f"\nEntries with missing end station IDs: {len(missing_end_station_id)}")
#print(missing_end_station_id.head())

df = df.dropna(subset=["end_station_id"])
df.info()

In [ ]:
df["ride_length_s"] = (pd.to_datetime(df["ended_at"]) - pd.to_datetime(df["started_at"])).dt.total_seconds()
df["ride_length_min"] = df["ride_length_s"] / 60

# Compute summary statistics for ride_length_s (mean, median, min, max, standard deviation)
summary_stats = df["ride_length_s"].agg(['mean', 'median', 'min', 'max', 'std'])
print("\nSummary statistics for ride_length_s:")
print(summary_stats)

plt.figure(figsize=(12, 5))
sns.boxplot(y="ride_length_min", data=df)
plt.ylabel("Ride Length (minutes)")
plt.title("Box Plot: Ride Length Distribution")

# Compute the number of rides of length greater than 4 hours
long_rides_count = (df["ride_length_s"] > 4 * 3600).sum()
print(f"\nNumber of rides longer than 4 hours: {long_rides_count}")

#Filter out rides longer than 4 hours
filtered_df = df[df["ride_length_s"] <= 4 * 3600]
plt.figure(figsize=(12, 5))
sns.boxplot(y="ride_length_min", data=filtered_df)
plt.ylabel("Ride Length (minutes)")
plt.title("Box Plot: Ride Length Distribution")

In [ ]:
# Count the number of rides for each rideable_type
rideable_type_counts = df["rideable_type"].value_counts()
print("\nNumber of rides for each rideable_type:")
print(rideable_type_counts)

# Compute the average ride length for each rideable_type
average_ride_length_by_type = df.groupby("rideable_type")["ride_length_s"].mean()
print("\nAverage ride length (in seconds) for each rideable_type:")
print(average_ride_length_by_type)

In [ ]:
# Compute the number of rides for each member_casual type
member_casual_counts = df["member_casual"].value_counts()
print("\nNumber of rides for each member_casual type:")
print(member_casual_counts)

# Compute the average ride length for each member_casual type
average_ride_length_by_member_casual = df.groupby("member_casual")["ride_length_s"].mean()
print("\nAverage ride length (in seconds) for each member_casual type:")
print(average_ride_length_by_member_casual)

In [ ]:
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

# Extract hour, day name, and date
df["start_hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.day_name()
df["date"] = df["started_at"].dt.date

# Create a new column for weekday/weekend
df["day_type"] = df["day_of_week"].apply(
    lambda x: "Weekday" if x in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"] else "Weekend"
)

# Count rides per (date, hour, day_type), then take the mean across dates
hourly_counts = (
    df.groupby(["date", "start_hour", "day_type"])
    .size()
    .reset_index(name="ride_count")
)
mean_rides = (
    hourly_counts.groupby(["start_hour", "day_type"])["ride_count"]
    .mean()
    .reset_index()
)

# Plot both in the same figure
plt.figure(figsize=(12, 5))
sns.barplot(x="start_hour", y="ride_count", hue="day_type", data=mean_rides)
plt.xlabel("Hour of Day")
plt.ylabel("Mean Number of Rides per Day")
plt.title("Mean Rides by Start Hour: Weekday vs Weekend")
plt.tight_layout()
plt.show()

During the weekdays, the number of rides is higher compared to the weekends. This could be due to people using the bike-sharing service for commuting to work or school during the weekdays.
We can clearly see it follows a multimodal distribution, with peaks around 7-8 AM and 5-6 PM, which likely correspond to the morning and evening rush hours when people are commuting to and from work or school. 

During the weekends, the distribution follows a normal distribution, with a peak around 1 PM, which could indicate that people are using the bike-sharing service for leisure activities during the weekends.

In [ ]:
# Count the number of rides for each start_station_name
station_counts = df["start_station_name"].value_counts()
print("\nNumber of rides for each start_station_name:")
print(station_counts.head(10))  # Print the top 10 stations

# Count the number of rides for each end_station_name
end_station_counts = df["end_station_name"].value_counts()
print("\nNumber of rides for each end_station_name:")
print(end_station_counts.head(10))  # Print the top 10 stations

Obviously, the more popular starting stations are also the more popular ending stations, which makes sense as they might be the more popular locations in the city.

We want instead to have a N x N matrix where N is the number of stations, and the value at (i, j) represents the number of rides from station i to station j. This way, we can analyze the flow of rides between stations more effectively.

In [ ]:
# Given the current DF, iterate over the rows and count the number of rides from each start_station_name to each end_station_name
station_flow = df.groupby(["start_station_name", "end_station_name"]).size().unstack(fill_value=0)

print("\nStation flow matrix shape:", station_flow.shape)
print("\nFirst 5x5 subset of the matrix:")
print(station_flow.iloc[:5, :5])
    

## Example of geospatial analysis

In [ ]:
# Heatmap of start stations
start_station_locations = df.groupby("start_station_name").agg({
    "start_lat": "first",
    "start_lng": "first",
    "ride_id": "count"
}).reset_index()
start_station_locations.rename(columns={"ride_id": "ride_count"}, inplace=True)

# Create a base map centered around the average location of the stations
map_center = [start_station_locations["start_lat"].mean(), start_station_locations["start_lng"].mean()]
bike_map = folium.Map(location=map_center, zoom_start=12)   
# Add heatmap layer for start stations
heat_data = [[row["start_lat"], row["start_lng"], row["ride_count"]] for index, row in start_station_locations.iterrows()]
HeatMap(heat_data).add_to(bike_map)
# Save the map to an HTML file
bike_map.save("start_station_heatmap.html")